In [7]:
# %% [markdown]
# # Uber NYC Ride-Hailing: Geographic Evolution 2018-2025
#
# Comparative spatial analysis of Uber pickup and dropoff patterns across New York City
# taxi zones, using K-means clustering on geographic coordinates, Lorenz curve concentration
# measures, and Local Indicators of Spatial Association (LISA) to quantify how ride-hailing
# demand has redistributed over seven years.

# %% [markdown]
# ---
# ## 1. Setup and Configuration

# %%
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist
from scipy.stats import ks_2samp, levene, entropy
from math import radians, cos, sin, asin, sqrt
import geopandas as gpd
import libpysal
from esda.moran import Moran, Moran_Local
import json
import gc
import os

# ── Paths ─────────────────────────────────────────────────────────────────
BASE_DIR = '/Users/leoss/Desktop/Portfolio/Website-/projects/uber/'
DATA_DIR = BASE_DIR + 'data/'
OUTPUT_DIR = BASE_DIR + 'outputs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH_2018 = DATA_DIR + 'fhv_tripdata_2018-01.parquet'
PATH_2025 = DATA_DIR + 'fhvhv_tripdata_2025-01.parquet'
PATH_CENTROIDS = DATA_DIR + 'zone_centroids.csv'
PATH_SHAPEFILE = DATA_DIR + 'taxi_zones/taxi_zones.shp'

# ── Analysis parameters ───────────────────────────────────────────────────
SAMPLE_SIZE = 20_000_000
N_CLUSTERS = 6
N_DEMAND_CLUSTERS = 4
K_NEAREST = 5
N_BOOTSTRAP = 1000

UBER_2018_BASES = ['B02512', 'B02598', 'B02617', 'B02682', 'B02764', 'B02765', 'B02835', 'B02836']
LYFT_2018_BASES = ['B02510']
UBER_2025_LICENSE = 'HV0003'
LYFT_2025_LICENSE = 'HV0005'

AIRPORT_ZONE_IDS = {132, 138, 1}
AIRPORT_LABELS = {132: 'JFK', 138: 'LaGuardia', 1: 'Newark (EWR)'}

# ── Unified style system (matches portfolio palette) ──────────────────
STYLE = {
    # Typography
    'font_family': 'Public Sans, -apple-system, BlinkMacSystemFont, sans-serif',
    'tick_size': 11,
    'axis_title_size': 13,
    'legend_size': 11,
    'annotation_size': 10,
    'title_color': '#1a2744',

    # Layout
    'template': 'plotly_white',
    'plot_bg': '#fafafa',
    'paper_bg': '#fafafa',
    'chart_height': 550,
    'margin_default': dict(l=60, r=40, t=20, b=50),
    'margin_map': dict(l=10, r=10, t=10, b=60),

    # Grid
    'grid_color': '#e5e7eb',
    'grid_width': 0.5,
    'zero_line_color': '#c9cfd6',

    # Hover
    'hover_font_size': 13,
    'hover_font_color': '#1a2744',

    # Year comparison palette
    'year_2018': '#4a6fa5',   # steel blue
    'year_2025': '#d4853b',   # warm amber

    # Cluster palette (6 geographic clusters)
    'cluster_colors': [
        '#c23a3a',   # muted red
        '#2e7d4a',   # forest green
        '#4a6fa5',   # steel blue
        '#d4853b',   # warm amber
        '#3d4f5f',   # dark slate
        '#8b5c3c',   # warm brown
    ],

    # Borough palette
    'borough_colors': {
        'Manhattan': '#4a6fa5', 'Brooklyn': '#2e7d4a', 'Queens': '#d4853b',
        'Bronx': '#c23a3a', 'Staten Island': '#3d4f5f', 'EWR': '#8b5c3c',
    },

    # LISA palette
    'lisa_colors': {
        'HH': '#c23a3a', 'LL': '#4a6fa5', 'HL': '#d4853b',
        'LH': '#7a8b99', 'ns': '#e8e8e8',
    },
    'lisa_labels': {
        'HH': 'High-High (hot spot)', 'LL': 'Low-Low (cold spot)',
        'HL': 'High-Low (outlier)', 'LH': 'Low-High (outlier)',
        'ns': 'Not significant',
    },

    # Map defaults
    'map_style': 'carto-positron-nolabels',
    'map_center': {'lat': 40.7128, 'lon': -73.9352},
    'map_zoom': 9.5,
}

# Demand profile colors (4 types)
DEMAND_COLORS = ['#3d4f5f', '#4a6fa5', '#d4853b', '#2e7d4a']
OPERATOR_COLORS = {'Uber': '#3d4f5f', 'Lyft': '#c23a3a', 'Other FHV': '#7a8b99'}

print(f"Configuration:")
print(f"  Sample size: {SAMPLE_SIZE:,} trips per year")
print(f"  Clusters: {N_CLUSTERS}")
print(f"  Output: {OUTPUT_DIR}")


# %% [markdown]
# ---
# ## 2. Helper Functions

# %%
# ── Layout helpers ────────────────────────────────────────────────────────

def base_layout(height=None, width=None, **kwargs):
    """Standard layout applied to every chart."""
    layout = dict(
        font=dict(family=STYLE['font_family']),
        template=STYLE['template'],
        plot_bgcolor=STYLE['plot_bg'],
        paper_bgcolor=STYLE['paper_bg'],
        autosize=True,
        margin=STYLE['margin_default'],
        hoverlabel=dict(
            font_size=STYLE['hover_font_size'],
            font_color=STYLE['hover_font_color'],
            bgcolor='white',
            bordercolor='#ccc',
        ),
    )
    if height:
        layout['height'] = height
    if width:
        layout['width'] = width
    layout.update(kwargs)
    return layout


def styled_axis(**kwargs):
    """Standard axis styling."""
    return dict(
        tickfont=dict(size=STYLE['tick_size']),
        title_font=dict(size=STYLE['axis_title_size']),
        gridcolor=STYLE['grid_color'],
        gridwidth=STYLE['grid_width'],
        **kwargs,
    )



# ── Revised export helpers ───────────────────────────────────────────────
def save_html(fig, filename, embed=False):
    """
    Save a Plotly figure as HTML.
    
    embed=False (default): embeddable fragment, loads Plotly from CDN.
    embed=True: bare div+script fragment, no Plotly JS included.
                Use this when inserting into a parent page that already
                loads Plotly once via <script src="...plotly.min.js">.
    """
    fig.write_html(
        OUTPUT_DIR + filename,
        full_html=False,
        include_plotlyjs='cdn' if not embed else False,
        config={'displayModeBar': False, 'responsive': True},
    )
    print(f"  Saved: {filename} ({'embed fragment' if embed else 'standalone'})")


def hex_to_rgba(hex_color, alpha=0.3):
    """Convert hex color to rgba string."""
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'


def add_bg_layer(fig):
    """Add grey background taxi zone layer to a map figure."""
    fig.add_trace(go.Choroplethmap(
        geojson=taxi_zones_geo_4326,
        locations=all_zone_ids,
        featureidkey='properties.LocationID',
        z=[1] * len(all_zone_ids),
        colorscale=[[0, '#e8e8e8'], [1, '#e8e8e8']],
        marker_opacity=0.4, marker_line_width=0.3, marker_line_color='#ccc',
        showscale=False, hoverinfo='skip',
    ))


def filter_geojson(zone_id_set):
    """Filter global taxi_zones_geo_4326 to a subset of zone IDs (as strings)."""
    return {
        'type': 'FeatureCollection',
        'features': [f for f in taxi_zones_geo_4326['features']
                     if f['properties']['LocationID'] in zone_id_set]
    }


# ── Geographic / statistical helpers ──────────────────────────────────────

def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance between two points in km."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return 2 * 6371 * asin(sqrt(a))


def haversine_km_vectorized(lat1, lon1, lat2, lon2):
    """Vectorized great-circle distance (numpy arrays, returns km)."""
    lat1, lon1, lat2, lon2 = (np.radians(x) for x in [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


def name_cluster_by_location(center_lat, center_lon, zone_centroids):
    """Name cluster based on nearest major zone."""
    dists = np.sqrt(
        (zone_centroids['latitude'] - center_lat) ** 2 +
        (zone_centroids['longitude'] - center_lon) ** 2
    )
    nearest = zone_centroids.iloc[dists.idxmin()]
    return f"{nearest['borough']}: {nearest['zone_name']}"


def match_clusters_by_proximity(centers_a, centers_b):
    """Match clusters by geographic proximity (greedy nearest, cityblock)."""
    dists = cdist(centers_a, centers_b, metric='cityblock')
    matches = {}
    used = set()
    pairs = sorted(
        [(i, j, dists[i, j]) for i in range(len(centers_a)) for j in range(len(centers_b))],
        key=lambda x: x[2]
    )
    for i, j, d in pairs:
        if i not in matches and j not in used:
            matches[i] = j
            used.add(j)
    return matches


def merge_zone_info(df, zone_col, centroids, prefix):
    """Merge zone centroid info onto a dataframe."""
    merged = df.merge(
        centroids[['zone_id', 'zone_name', 'borough', 'latitude', 'longitude']],
        left_on=zone_col, right_on='zone_id', how='left'
    )
    return merged.rename(columns={
        'zone_id': f'{prefix}_zone_id',
        'zone_name': f'{prefix}_zone_name',
        'borough': f'{prefix}_borough',
        'latitude': f'{prefix}_lat',
        'longitude': f'{prefix}_lon',
    })


def short_label(name):
    """Extract borough from 'Borough: Zone Name' format."""
    return name.split(':')[0].strip()


def make_short_labels(cluster_names, n_clusters):
    """Create short labels, appending (2), (3) for duplicate boroughs."""
    labels = {}
    seen = {}
    for i in range(n_clusters):
        base = short_label(cluster_names[i])
        if base in seen:
            seen[base] += 1
            labels[i] = f"{base} ({seen[base]})"
        else:
            seen[base] = 1
            labels[i] = base
    return labels


def lorenz_data(zone_counts_series):
    """Return (cumulative share of zones, cumulative share of trips)."""
    vals = zone_counts_series.sort_values().values
    cum_zones = np.arange(1, len(vals) + 1) / len(vals)
    cum_trips = np.cumsum(vals) / vals.sum()
    return cum_zones, cum_trips


def gini_from_lorenz(cum_zones, cum_trips):
    """Gini coefficient via trapezoidal integration under the Lorenz curve."""
    _trapz = getattr(np, 'trapezoid', np.trapz)  # numpy <2.0 compat
    return 1 - 2 * _trapz(cum_trips, cum_zones)


def mismatch_ratios(df, pu_col='PU_zone_id', do_col='DO_zone_id'):
    """Compute PU/(PU+DO) ratio per zone. 0.5 = perfectly balanced."""
    pu = df.groupby(pu_col).size().rename('pu')
    do = df.dropna(subset=[do_col]).groupby(do_col).size().rename('do')
    combined = pd.concat([pu, do], axis=1).fillna(0)
    combined['ratio'] = combined['pu'] / (combined['pu'] + combined['do'])
    combined['total'] = combined['pu'] + combined['do']
    return combined


def hourly_profile(df, filter_col=None, filter_val=None):
    """Return normalized 24-bin hourly distribution."""
    if filter_col is not None:
        df = df[df[filter_col] == filter_val]
    counts = df.groupby('hour').size().reindex(range(24), fill_value=0)
    total = counts.sum()
    if total == 0:
        return np.ones(24) / 24
    return (counts / total).values


def jsd(p, q):
    """Jensen-Shannon divergence (symmetric, bounded [0, 1] with log2)."""
    eps = 1e-12
    p = np.array(p, dtype=float) + eps
    q = np.array(q, dtype=float) + eps
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    return 0.5 * (entropy(p, m, base=2) + entropy(q, m, base=2))


def pivot_od(df, n):
    """Build cluster-to-cluster OD share matrix (%)."""
    valid = df.dropna(subset=['PU_cluster', 'DO_cluster']).copy()
    valid['PU_cluster'] = valid['PU_cluster'].astype(int)
    valid['DO_cluster'] = valid['DO_cluster'].astype(int)
    od = valid.groupby(['PU_cluster', 'DO_cluster']).size().reset_index(name='trips')
    total = od['trips'].sum()
    matrix = np.zeros((n, n))
    for _, r in od.iterrows():
        if int(r['PU_cluster']) < n and int(r['DO_cluster']) < n:
            matrix[int(r['PU_cluster']), int(r['DO_cluster'])] = 100 * r['trips'] / total
    return matrix


def compute_zone_localization(df, nearest_zones_dict,
                               pu_zone_col='PU_zone_id', do_zone_col='DO_zone_id'):
    """For each PU zone, fraction of trips where DO is in same or K nearest zones."""
    valid = df.dropna(subset=[pu_zone_col, do_zone_col]).copy()
    valid[pu_zone_col] = valid[pu_zone_col].astype(int)
    valid[do_zone_col] = valid[do_zone_col].astype(int)
    od_counts = valid.groupby([pu_zone_col, do_zone_col]).size().reset_index(name='trips')
    od_counts['is_local'] = od_counts.apply(
        lambda r: int(r[do_zone_col]) in nearest_zones_dict.get(int(r[pu_zone_col]), set()),
        axis=1
    )
    total_by_zone = od_counts.groupby(pu_zone_col)['trips'].sum().rename('total')
    local_by_zone = od_counts[od_counts['is_local']].groupby(pu_zone_col)['trips'].sum().rename('local')
    zone_local = pd.concat([total_by_zone, local_by_zone], axis=1).fillna(0)
    zone_local['local_share'] = 100 * zone_local['local'] / zone_local['total']
    return zone_local


def bootstrap_gini(df, zone_col='PU_zone_id', n_boot=N_BOOTSTRAP):
    """Bootstrap Gini by resampling zone-level trip counts."""
    _trapz = getattr(np, 'trapezoid', np.trapz)
    zone_counts = df.groupby(zone_col).size().values
    n_zones = len(zone_counts)
    rng = np.random.RandomState(42)
    ginis = np.empty(n_boot)
    for b in range(n_boot):
        sampled = rng.choice(zone_counts, size=n_zones, replace=True)
        sorted_counts = np.sort(sampled)
        cz = np.arange(1, n_zones + 1) / n_zones
        ct = np.cumsum(sorted_counts) / sorted_counts.sum()
        ginis[b] = 1 - 2 * _trapz(ct, cz)
    return ginis


# %% [markdown]
# ---
# ## 3. Load Zone Centroids, Shapefile, and Spatial Weights

# %%
zone_centroids = pd.read_csv(PATH_CENTROIDS)
print(f"Loaded {len(zone_centroids)} zone centroids")
gdf_raw = gpd.read_file(PATH_SHAPEFILE).to_crs(epsg=4326)
gdf_raw['geometry'] = gdf_raw['geometry'].simplify(tolerance=0.0001, preserve_topology=True)
taxi_zones_geo_4326 = json.loads(gdf_raw.to_json())



for f in taxi_zones_geo_4326['features']:
    f['properties']['LocationID'] = str(int(f['properties']['LocationID']))
all_zone_ids = [f['properties']['LocationID'] for f in taxi_zones_geo_4326['features']]
print(f"Loaded {len(all_zone_ids)} taxi zone geometries")

# Spatial weights (used by LISA and Moran's I) - computed once
w = libpysal.weights.KNN.from_dataframe(gdf_raw, k=6)
w.transform = 'r'
gdf_base = gdf_raw.copy()
gdf_base['LocationID_int'] = gdf_base['LocationID'].astype(int)

# Zone centroid lookup (used by many charts)
zc_lookup = zone_centroids.drop_duplicates(subset='zone_id').set_index('zone_id')

# K-nearest zones for localization metric
zc_deduped = zone_centroids.drop_duplicates(subset='zone_id').copy().set_index('zone_id')
zone_ids_all_int = zc_deduped.index.values
zone_coords_arr = zc_deduped[['latitude', 'longitude']].values
zone_dist_matrix = cdist(zone_coords_arr, zone_coords_arr, metric='euclidean')
nearest_zones = {}
for i, zid in enumerate(zone_ids_all_int):
    sorted_idx = np.argsort(zone_dist_matrix[i])
    nearest_zones[zid] = set([zone_ids_all_int[j] for j in sorted_idx[1:K_NEAREST + 1]]) | {zid}

print(f"Spatial weights: KNN k=6, {len(gdf_raw)} geometries")
print(f"K-nearest zones: K={K_NEAREST}, {len(nearest_zones)} zones")


# %% [markdown]
# ---
# ## 4. Load and Process Trip Data (Once)
#
# Both parquet files are read exactly once. Filtering, sampling, temporal features,
# zone merging, and K-means clustering happen here and nowhere else.

# %%
def load_year(path, year, filter_col, filter_val, pu_col, do_col):
    """Load, filter to Uber, sample, add temporal features, merge zones, cluster.
    Memory-conscious: deletes intermediates at each step."""
    print(f"\n[Loading {year}] {os.path.basename(path)}")

    # ── Row count (zero-column read, trivial memory) ─────────────────────
    total = pq.read_table(path, columns=[]).num_rows
    print(f"  Total rows: {total:,}")

    # ── Read needed columns, convert to pandas, free arrow table ─────────
    cols = ['pickup_datetime', pu_col, do_col, filter_col]
    table = pq.read_table(path, columns=cols)
    df_full = table.to_pandas()
    del table
    gc.collect()

    # ── Filter to Uber, free full dataframe ──────────────────────────────
    if isinstance(filter_val, list):
        uber_mask = df_full[filter_col].isin(filter_val)
    else:
        uber_mask = df_full[filter_col] == filter_val
    uber_count = uber_mask.sum()
    print(f"  Uber trips: {uber_count:,} ({100 * uber_count / total:.1f}%)")

    df = df_full.loc[uber_mask].copy()
    del df_full, uber_mask
    gc.collect()

    # ── Sample ───────────────────────────────────────────────────────────
    if len(df) > SAMPLE_SIZE:
        df = df.sample(n=SAMPLE_SIZE, random_state=42)

    # ── Temporal features ────────────────────────────────────────────────
    df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
    df['hour'] = df['pickup_datetime'].dt.hour
    df['day_of_week'] = df['pickup_datetime'].dt.dayofweek
    df['day_name'] = df['pickup_datetime'].dt.day_name()
    # Drop columns no longer needed (saves ~200MB for 15M rows)
    df = df.drop(columns=['pickup_datetime', filter_col])

    # ── Pickup zone merge ────────────────────────────────────────────────
    df = df.dropna(subset=[pu_col])
    df[pu_col] = df[pu_col].astype(int)
    df = merge_zone_info(df, pu_col, zone_centroids, 'PU')
    df = df.dropna(subset=['PU_lat', 'PU_lon'])

    # ── Dropoff zone merge (fix dtype warning) ───────────────────────────
    df[do_col] = pd.to_numeric(df[do_col], errors='coerce')
    do_notna = df[do_col].notna()
    df.loc[do_notna, do_col] = df.loc[do_notna, do_col].astype(float).astype('Int64')
    df = merge_zone_info(df, do_col, zone_centroids, 'DO')

    # ── K-means clustering on pickup coordinates ─────────────────────────
    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    df['PU_cluster'] = kmeans.fit_predict(df[['PU_lat', 'PU_lon']].values)
    cnames = {i: name_cluster_by_location(*kmeans.cluster_centers_[i], zone_centroids)
              for i in range(N_CLUSTERS)}
    df['PU_cluster_name'] = df['PU_cluster'].map(cnames)

    # ── Predict dropoff clusters ─────────────────────────────────────────
    do_mask = df['DO_lat'].notna() & df['DO_lon'].notna()
    df.loc[do_mask, 'DO_cluster'] = kmeans.predict(
        df.loc[do_mask, ['DO_lat', 'DO_lon']].values)
    df['DO_cluster_name'] = df['DO_cluster'].map(cnames)

    print(f"  Final sample: {len(df):,} trips")
    for i in range(N_CLUSTERS):
        count = (df['PU_cluster'] == i).sum()
        pct = 100 * count / len(df)
        print(f"    {cnames[i]}: {count:>10,} ({pct:>5.1f}%)")

    gc.collect()
    return df, kmeans, cnames, total, uber_count


# Load both years (with gc between them)
df_2018, kmeans_2018, cnames_2018, total_2018, uber_n_2018 = load_year(
    PATH_2018, 2018, 'dispatching_base_num', UBER_2018_BASES,
    'PUlocationID', 'DOlocationID'
)
gc.collect()

df_2025, kmeans_2025, cnames_2025, total_2025, uber_n_2025 = load_year(
    PATH_2025, 2025, 'hvfhs_license_num', UBER_2025_LICENSE,
    'PULocationID', 'DOLocationID'
)
gc.collect()

# ── Trip distances (computed after both loads to avoid peak memory) ───────
print("\nComputing trip distances...")
for label, df in [('2018', df_2018), ('2025', df_2025)]:
    valid = df['DO_lat'].notna() & df['DO_lon'].notna()
    df.loc[valid, 'trip_dist_km'] = haversine_km_vectorized(
        df.loc[valid, 'PU_lat'].values, df.loc[valid, 'PU_lon'].values,
        df.loc[valid, 'DO_lat'].values, df.loc[valid, 'DO_lon'].values)
    n = valid.sum()
    print(f"  {label}: {n:,} trips with distances")


# %% [markdown]
# ---
# ## 5. Cross-Year Cluster Matching and Derived Quantities
#
# Everything computed here is reused across many charts.

# %%
# ── Cluster matching ─────────────────────────────────────────────────────
centers_2018 = kmeans_2018.cluster_centers_
centers_2025 = kmeans_2025.cluster_centers_
cluster_matches = match_clusters_by_proximity(centers_2018, centers_2025)

print("Cluster matches (2018 -> 2025):")
for i18, i25 in sorted(cluster_matches.items()):
    dist = haversine_km(*centers_2018[i18], *centers_2025[i25])
    print(f"  {cnames_2018[i18]} -> {cnames_2025[i25]} ({dist:.2f} km)")

# ── Short labels ─────────────────────────────────────────────────────────
short_labels_2018 = make_short_labels(cnames_2018, N_CLUSTERS)
short_labels_2025 = make_short_labels(cnames_2025, N_CLUSTERS)

# ── Color maps (matched clusters share same color) ───────────────────────
cluster_color_map_2018 = {
    cnames_2018[i]: STYLE['cluster_colors'][i % len(STYLE['cluster_colors'])]
    for i in range(N_CLUSTERS)
}
cluster_color_map_2025 = {
    cnames_2025[i25]: STYLE['cluster_colors'][i18 % len(STYLE['cluster_colors'])]
    for i18, i25 in cluster_matches.items()
}

# ── Centroid shift data ──────────────────────────────────────────────────
shift_data = []
for i18, i25 in cluster_matches.items():
    lat1, lon1 = centers_2018[i18]
    lat2, lon2 = centers_2025[i25]
    shift_data.append({
        'idx_18': i18, 'idx_25': i25,
        'name_18': cnames_2018[i18], 'name_25': cnames_2025[i25],
        'dist_km': haversine_km(lat1, lon1, lat2, lon2)
    })
avg_shift = np.mean([s['dist_km'] for s in shift_data])

# ── Borough aggregates ───────────────────────────────────────────────────
borough_2018 = df_2018.groupby('PU_borough').size().reset_index(name='trips')
borough_2025 = df_2025.groupby('PU_borough').size().reset_index(name='trips')
borough_2018['pct'] = 100 * borough_2018['trips'] / borough_2018['trips'].sum()
borough_2025['pct'] = 100 * borough_2025['trips'] / borough_2025['trips'].sum()
borough_df = borough_2018[['PU_borough', 'pct']].rename(columns={'pct': 'pct_2018'}).merge(
    borough_2025[['PU_borough', 'pct']].rename(columns={'pct': 'pct_2025'}), on='PU_borough'
)

# ── Zone-level pickup shares ─────────────────────────────────────────────
pu_counts_2018 = df_2018.groupby(['PU_zone_id', 'PU_zone_name', 'PU_borough']).size().reset_index(name='count')
pu_counts_2025 = df_2025.groupby(['PU_zone_id', 'PU_zone_name', 'PU_borough']).size().reset_index(name='count')
pu_counts_2018['share'] = 100 * pu_counts_2018['count'] / pu_counts_2018['count'].sum()
pu_counts_2025['share'] = 100 * pu_counts_2025['count'] / pu_counts_2025['count'].sum()

zone_change = pu_counts_2018[['PU_zone_id', 'PU_zone_name', 'PU_borough', 'share']].rename(
    columns={'share': 'share_2018', 'PU_zone_name': 'zone_name', 'PU_borough': 'borough'}
).merge(
    pu_counts_2025[['PU_zone_id', 'share']].rename(columns={'share': 'share_2025'}),
    on='PU_zone_id', how='outer'
).fillna(0)
zone_change['share_change'] = zone_change['share_2025'] - zone_change['share_2018']
zone_change['zone_id'] = zone_change['PU_zone_id'].astype(float).astype(int).astype(str)
cap_pu = zone_change['share_change'].abs().quantile(0.95)
zone_change['share_change_capped'] = zone_change['share_change'].clip(-cap_pu, cap_pu)
zone_change['zone_name'] = zone_change['zone_id'].astype(int).map(zc_lookup['zone_name'])
zone_change['borough'] = zone_change['zone_id'].astype(int).map(zc_lookup['borough'])

# ── Dropoff share change ─────────────────────────────────────────────────
do_counts_2018 = df_2018.dropna(subset=['DO_zone_id']).groupby('DO_zone_id').size().reset_index(name='count_2018')
do_counts_2025 = df_2025.dropna(subset=['DO_zone_id']).groupby('DO_zone_id').size().reset_index(name='count_2025')
do_comparison = do_counts_2018.merge(do_counts_2025, left_on='DO_zone_id', right_on='DO_zone_id', how='outer').fillna(0)
do_comparison['share_2018'] = 100 * do_comparison['count_2018'] / do_comparison['count_2018'].sum()
do_comparison['share_2025'] = 100 * do_comparison['count_2025'] / do_comparison['count_2025'].sum()
do_comparison['share_change'] = do_comparison['share_2025'] - do_comparison['share_2018']
do_comparison['zone_id'] = do_comparison['DO_zone_id'].astype(float).astype(int).astype(str)
do_comparison['zone_name'] = do_comparison['DO_zone_id'].astype(int).map(zc_lookup['zone_name'])
do_comparison['borough'] = do_comparison['DO_zone_id'].astype(int).map(zc_lookup['borough'])
cap_do = do_comparison['share_change'].abs().quantile(0.95)
do_comparison['share_change_capped'] = do_comparison['share_change'].clip(-cap_do, cap_do)

# ── Gini + Lorenz ────────────────────────────────────────────────────────
pu_zone_counts_2018 = df_2018.groupby('PU_zone_id').size()
pu_zone_counts_2025 = df_2025.groupby('PU_zone_id').size()
cz_18, ct_18 = lorenz_data(pu_zone_counts_2018)
cz_25, ct_25 = lorenz_data(pu_zone_counts_2025)
gini_18 = gini_from_lorenz(cz_18, ct_18)
gini_25 = gini_from_lorenz(cz_25, ct_25)
print(f"\nGini: {gini_18:.4f} (2018) -> {gini_25:.4f} (2025)")

# ── Bootstrap Gini CIs ──────────────────────────────────────────────────
print("Bootstrapping Gini CIs...")
gini_boot_2018 = bootstrap_gini(df_2018)
gini_boot_2025 = bootstrap_gini(df_2025)
ci_2018 = np.percentile(gini_boot_2018, [2.5, 97.5])
ci_2025 = np.percentile(gini_boot_2025, [2.5, 97.5])
print(f"  2018: {gini_18:.4f} [{ci_2018[0]:.4f}, {ci_2018[1]:.4f}]")
print(f"  2025: {gini_25:.4f} [{ci_2025[0]:.4f}, {ci_2025[1]:.4f}]")

# ── Moran's I ────────────────────────────────────────────────────────────
def compute_morans(df_year):
    zone_counts = df_year.groupby('PU_zone_id').size()
    gdf_tmp = gdf_base.copy()
    gdf_tmp['trips'] = gdf_tmp['LocationID_int'].map(zone_counts).fillna(0)
    gdf_tmp['trip_share'] = 100 * gdf_tmp['trips'] / gdf_tmp['trips'].sum()
    mi = Moran(gdf_tmp['trip_share'].values, w)
    return mi.I, mi.p_sim

mi_2018, mi_p_2018 = compute_morans(df_2018)
mi_2025, mi_p_2025 = compute_morans(df_2025)
print(f"Moran's I: {mi_2018:.4f} (p={mi_p_2018:.4f}) -> {mi_2025:.4f} (p={mi_p_2025:.4f})")

# ── Intra-cluster trip shares ────────────────────────────────────────────
intra_2018 = df_2018.dropna(subset=['PU_cluster', 'DO_cluster'])
intra_2018_pct = 100 * (intra_2018['PU_cluster'] == intra_2018['DO_cluster']).mean()
intra_2025 = df_2025.dropna(subset=['PU_cluster', 'DO_cluster'])
intra_2025_pct = 100 * (intra_2025['PU_cluster'] == intra_2025['DO_cluster']).mean()
print(f"Intra-cluster trips: {intra_2018_pct:.1f}% -> {intra_2025_pct:.1f}%")

# ── Mismatch ratios + tests ─────────────────────────────────────────────
mr_2018 = mismatch_ratios(df_2018)
mr_2025 = mismatch_ratios(df_2025)
ks_stat, ks_p = ks_2samp(mr_2018['ratio'].dropna(), mr_2025['ratio'].dropna())
lev_stat, lev_p = levene(mr_2018['ratio'].dropna(), mr_2025['ratio'].dropna())
print(f"KS mismatch: D={ks_stat:.4f}, p={ks_p:.2e}")
print(f"Levene: W={lev_stat:.4f}, p={lev_p:.2e}")

# ── OD matrices ──────────────────────────────────────────────────────────
od_matrix_2018 = pivot_od(df_2018, N_CLUSTERS)
od_matrix_2025 = pivot_od(df_2025, N_CLUSTERS)
od_2025_aligned = np.zeros_like(od_matrix_2025)
for i18, i25 in cluster_matches.items():
    for j18, j25 in cluster_matches.items():
        od_2025_aligned[i18, j18] = od_matrix_2025[i25, j25]
od_diff = od_2025_aligned - od_matrix_2018

# ── Hourly profiles and JSD ──────────────────────────────────────────────
h18_pct = df_2018.groupby('hour').size()
h25_pct = df_2025.groupby('hour').size()
h18_pct = 100 * h18_pct / h18_pct.sum()
h25_pct = 100 * h25_pct / h25_pct.sum()

agg_jsd = jsd(hourly_profile(df_2018), hourly_profile(df_2025))
cluster_jsds = []
for i in range(N_CLUSTERS):
    i25 = cluster_matches[i]
    d = jsd(hourly_profile(df_2018, 'PU_cluster', i),
            hourly_profile(df_2025, 'PU_cluster', i25))
    cluster_jsds.append({'cluster': short_labels_2018[i], 'jsd': d, 'idx': i})
cjsd_df = pd.DataFrame(cluster_jsds)
print(f"Aggregate hourly JSD: {agg_jsd:.6f}")

# ── Localization ─────────────────────────────────────────────────────────
print("Computing zone localization...")
local_2018 = compute_zone_localization(df_2018, nearest_zones)
local_2025 = compute_zone_localization(df_2025, nearest_zones)
mean_local_2018 = local_2018['local_share'].mean()
mean_local_2025 = local_2025['local_share'].mean()
print(f"  Localization: {mean_local_2018:.1f}% -> {mean_local_2025:.1f}%")

# ── Trip distances (filtered) ────────────────────────────────────────────
d18_valid = df_2018[(df_2018['trip_dist_km'] > 0) & (df_2018['trip_dist_km'] < 100)]['trip_dist_km']
d25_valid = df_2025[(df_2025['trip_dist_km'] > 0) & (df_2025['trip_dist_km'] < 100)]['trip_dist_km']
print(f"Trip distances: median {d18_valid.median():.2f} km (2018) vs {d25_valid.median():.2f} km (2025)")

# ── Zone-level cluster assignments (for maps) ────────────────────────────
def zone_cluster_assignment(df):
    zc = df.groupby('PU_zone_id').agg({
        'PU_cluster': lambda x: x.mode()[0],
        'PU_cluster_name': lambda x: x.mode()[0],
        'PU_zone_name': 'first',
        'PU_borough': 'first'
    }).reset_index()
    zc.columns = ['zone_id', 'cluster', 'cluster_name', 'zone_name', 'borough']
    zc['zone_id'] = zc['zone_id'].astype(float).astype(int).astype(str)
    return zc

zone_clusters_2018 = zone_cluster_assignment(df_2018)
zone_clusters_2025 = zone_cluster_assignment(df_2025)

print("\n" + "=" * 70)
print("ALL PRECOMPUTATIONS COMPLETE")
print("=" * 70)
# %% [markdown]
# ---
# ## 6. Cluster Maps (2018 + 2025)

# %%
for year_label, zc_df, color_map, cnames, centers, kmeans_obj in [
    ('2018', zone_clusters_2018, cluster_color_map_2018, cnames_2018, centers_2018, kmeans_2018),
    ('2025', zone_clusters_2025, cluster_color_map_2025, cnames_2025, centers_2025, kmeans_2025),
]:
    print(f"[{year_label}] Cluster map...")
    fig = go.Figure()
    add_bg_layer(fig)

    zone_ids_set = set(zc_df['zone_id'].values)
    fgeo = filter_geojson(zone_ids_set)

    fig_tmp = px.choropleth_map(
        zc_df, geojson=fgeo,
        locations='zone_id', featureidkey='properties.LocationID',
        color='cluster_name', color_discrete_map=color_map,
        map_style=STYLE['map_style'], zoom=STYLE['map_zoom'],
        center=STYLE['map_center'], opacity=0.95,
        hover_data={'zone_id': False, 'zone_name': True, 'borough': True, 'cluster_name': True},
        labels={'cluster_name': 'Cluster'}
    )
    fig_tmp.update_traces(marker=dict(line=dict(width=0.8, color='rgba(255,255,255,0.8)')))
    for trace in fig_tmp.data:
        fig.add_trace(trace)

    for i, (lat, lon) in enumerate(centers):
        fig.add_trace(go.Scattermap(
            lat=[lat], lon=[lon], mode='markers+text',
            marker=dict(size=0, color='black', opacity=0.85),
            text=str(i), textfont=dict(size=10, color='white', family='Arial Black'),
            textposition='middle center', name=cnames[i], showlegend=False,
            hovertemplate=f'<b>Cluster {i}</b><br>{cnames[i]}<extra></extra>',
        ))

    fig.update_layout(
        **base_layout(height=650, width=1100, margin=STYLE['margin_map']),
        legend=dict(
            title="Clusters", yanchor="top", y=0.99, xanchor="left", x=0.01,
            bgcolor="rgba(255,255,255,0.95)", bordercolor="#dde1e7", borderwidth=1,
            font=dict(size=STYLE['legend_size'], family=STYLE['font_family']),
        ),
        map=dict(style=STYLE['map_style'], center=STYLE['map_center'], zoom=STYLE['map_zoom']),
    )
    save_html(fig, f'cluster_map_{year_label}.html')


# %% [markdown]
# ---
# ## 7. Cluster Centroid Shifts Map

# %%
print("Centroid shifts map...")
fig_shift = go.Figure()
add_bg_layer(fig_shift)

for sa in shift_data:
    lat1, lon1 = centers_2018[sa['idx_18']]
    lat2, lon2 = centers_2025[sa['idx_25']]
    fig_shift.add_trace(go.Scattermap(
        lat=[lat1, lat2], lon=[lon1, lon2],
        mode='lines', line=dict(width=3, color='#333'),
        showlegend=False,
        hovertemplate=(f"<b>{short_label(sa['name_18'])}</b><br>"
                       f"Shift: {sa['dist_km']:.2f} km<extra></extra>")))
    mid_lat = (lat1 + lat2) / 2
    mid_lon = (lon1 + lon2) / 2
    fig_shift.add_trace(go.Scattermap(
        lat=[mid_lat + (lat2 - lat1) * 0.15], lon=[mid_lon + (lon2 - lon1) * 0.15],
        mode='markers', marker=dict(size=8, color='#333', symbol='triangle'),
        showlegend=False, hoverinfo='skip'))

for i in range(N_CLUSTERS):
    fig_shift.add_trace(go.Scattermap(
        lat=[centers_2018[i, 0]], lon=[centers_2018[i, 1]],
        mode='markers+text',
        marker=dict(size=14, color=STYLE['cluster_colors'][i], opacity=0.95, symbol='circle'),
        text=short_labels_2018[i],
        textfont=dict(size=9, color='#333', family=STYLE['font_family']),
        textposition='top center',
        name=f'2018: {short_labels_2018[i]}',
        legendgroup='2018', legendgrouptitle_text='2018',
        hovertemplate=f'<b>2018: {short_labels_2018[i]}</b><extra></extra>'))

for i in range(N_CLUSTERS):
    matched_18 = [k for k, v in cluster_matches.items() if v == i]
    color_idx = matched_18[0] if matched_18 else i
    fig_shift.add_trace(go.Scattermap(
        lat=[centers_2025[i, 0]], lon=[centers_2025[i, 1]],
        mode='markers',
        marker=dict(size=14, color=STYLE['cluster_colors'][color_idx], opacity=0.95, symbol='square'),
        name=f'2025: {short_labels_2025[i]}',
        legendgroup='2025', legendgrouptitle_text='2025',
        hovertemplate=f'<b>2025: {short_labels_2025[i]}</b><extra></extra>'))

fig_shift.update_layout(
    **base_layout(height=700, width=1100, margin=STYLE['margin_map']),
    map_style=STYLE['map_style'], map_zoom=9.8,
    map_center=dict(lat=40.72, lon=-73.94),
    legend=dict(
        yanchor='top', y=0.99, xanchor='left', x=0.01,
        bgcolor='rgba(255,255,255,0.95)', bordercolor='#dde1e7', borderwidth=1,
        font=dict(size=11, family=STYLE['font_family']),
        itemsizing='constant'))
save_html(fig_shift, 'fix_cluster_shifts.html')


# %% [markdown]
# ---
# ## 8. Pickup Density Change Map

# %%
print("Pickup density change map...")
fig_pu = go.Figure()
add_bg_layer(fig_pu)

zids_change = set(zone_change['zone_id'].values)
fgeo_change = filter_geojson(zids_change)

fig_pu.add_trace(go.Choroplethmap(
    geojson=fgeo_change,
    locations=zone_change['zone_id'],
    featureidkey='properties.LocationID',
    z=zone_change['share_change_capped'],
    colorscale='RdBu', zmid=0,
    marker_opacity=0.85, marker_line_width=0.5,
    marker_line_color='rgba(255,255,255,0.6)',
    colorbar=dict(
        title=dict(text='Change (pp)', font=dict(family=STYLE['font_family'])),
        ticksuffix=' pp', x=0.99,
        tickfont=dict(family=STYLE['font_family']),
    ),
    customdata=np.column_stack([
        zone_change['zone_name'].fillna('Unknown'),
        zone_change['borough'].fillna('Unknown'),
        zone_change['share_change']
    ]),
    hovertemplate=(
        '<b>%{customdata[0]}</b> (%{customdata[1]})<br>'
        'Change: %{customdata[2]:.2f} pp<extra></extra>'
    ),
))
fig_pu.update_layout(
    **base_layout(height=700, width=1100, margin=STYLE['margin_map']),
    map_style=STYLE['map_style'], map_zoom=STYLE['map_zoom'],
    map_center=dict(**STYLE['map_center']),
)
save_html(fig_pu, 'pickup_density_change.html')


# %% [markdown]
# ---
# ## 9. Borough Analysis (% normalized)

# %%
print("Borough analysis...")
fig = go.Figure()
fig.add_trace(go.Bar(x=borough_df['PU_borough'], y=borough_df['pct_2018'], name='2018',
    marker_color=STYLE['year_2018'],
    text=[f"{v:.1f}%" for v in borough_df['pct_2018']], textposition='outside',
    hovertemplate='%{x}<br>2018: %{y:.1f}%<extra></extra>'))
fig.add_trace(go.Bar(x=borough_df['PU_borough'], y=borough_df['pct_2025'], name='2025',
    marker_color=STYLE['year_2025'],
    text=[f"{v:.1f}%" for v in borough_df['pct_2025']], textposition='outside',
    hovertemplate='%{x}<br>2025: %{y:.1f}%<extra></extra>'))
fig.update_layout(**base_layout(),
    xaxis=styled_axis(title_text='Borough'),
    yaxis=styled_axis(title_text='Share of Pickups (%)', range=[0, 80]),
    barmode='group',
    legend=dict(x=0.85, y=0.98, font=dict(family=STYLE['font_family'])))
save_html(fig, 'fix_borough_pct.html')


# %% [markdown]
# ---
# ## 10. Hourly Patterns

# %%
print("Hourly patterns...")
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=h18_pct.index, y=h18_pct.values, name='2018', mode='lines',
    line=dict(color=STYLE['year_2018'], width=2.5, shape='spline'),
    fill='tozeroy', fillcolor='rgba(74,111,165,0.1)',
    hovertemplate='Hour %{x}:00<br>%{y:.1f}% of trips<extra>2018</extra>'))
fig.add_trace(go.Scatter(
    x=h25_pct.index, y=h25_pct.values, name='2025', mode='lines',
    line=dict(color=STYLE['year_2025'], width=2.5, shape='spline'),
    fill='tozeroy', fillcolor='rgba(212,133,59,0.1)',
    hovertemplate='Hour %{x}:00<br>%{y:.1f}% of trips<extra>2025</extra>'))
fig.update_layout(**base_layout(),
    xaxis=styled_axis(title_text='Hour of Day', dtick=2),
    yaxis=styled_axis(title_text='Share of Daily Trips (%)'),
    legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])))
save_html(fig, 'fix_hourly_patterns.html')


# %% [markdown]
# ---
# ## 11. Per-Cluster Temporal Change Heatmaps (2x3 grid)

# %%
print("Per-cluster temporal change heatmaps...")
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
fig_heat = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f"{short_labels_2018[i]}" for i in range(N_CLUSTERS)],
    shared_xaxes=True, shared_yaxes=True,
    vertical_spacing=0.08, horizontal_spacing=0.04)

for i in range(N_CLUSTERS):
    row = i // 3 + 1
    col = i % 3 + 1
    c18 = df_2018[df_2018['PU_cluster'] == i]
    piv18 = c18.groupby(['day_name', 'hour']).size().reset_index(name='trips')
    piv18 = piv18.pivot(index='day_name', columns='hour', values='trips').reindex(day_order).fillna(0)
    piv18_pct = 100 * piv18 / piv18.sum().sum()

    i25 = cluster_matches[i]
    c25 = df_2025[df_2025['PU_cluster'] == i25]
    piv25 = c25.groupby(['day_name', 'hour']).size().reset_index(name='trips')
    piv25 = piv25.pivot(index='day_name', columns='hour', values='trips').reindex(day_order).fillna(0)
    piv25_pct = 100 * piv25 / piv25.sum().sum()

    diff = piv25_pct.values - piv18_pct.values
    fig_heat.add_trace(go.Heatmap(
        z=diff, x=list(range(24)), y=day_order,
        colorscale='RdBu', zmid=0,
        showscale=(i == N_CLUSTERS - 1),
        colorbar=dict(
            title=dict(text='Change (pp)'), ticksuffix=' pp', x=1.02,
            tickfont=dict(family=STYLE['font_family'])) if i == N_CLUSTERS - 1 else None,
        hovertemplate='<b>%{y}, %{x}:00</b><br>Change: %{z:.3f} pp<extra></extra>',
    ), row=row, col=col)

fig_heat.update_layout(**base_layout())
fig_heat.update_xaxes(title_text='Hour', row=2)
fig_heat.update_yaxes(tickfont=dict(size=9))
save_html(fig_heat, 'fix_cluster_heatmap_changes.html')


# %% [markdown]
# ---
# ## 12. Hourly Composition by Cluster (stacked bar, matched)

# %%
print("Hourly composition by cluster...")
fig_comp = make_subplots(rows=1, cols=2,
    subplot_titles=('2018', '2025'), horizontal_spacing=0.08)
cluster_order = sorted(range(N_CLUSTERS), key=lambda i: -(df_2018['PU_cluster'] == i).sum())

for col_idx, (df_year, year_label) in enumerate([(df_2018, '2018'), (df_2025, '2025')], 1):
    hourly_by_cluster = df_year.groupby(['hour', 'PU_cluster']).size().reset_index(name='trips')
    hourly_total = df_year.groupby('hour').size().reset_index(name='total')
    hourly_by_cluster = hourly_by_cluster.merge(hourly_total, on='hour')
    hourly_by_cluster['share'] = 100 * hourly_by_cluster['trips'] / hourly_by_cluster['total']

    for c in cluster_order:
        cluster_idx = c if year_label == '2018' else cluster_matches[c]
        label = short_labels_2018[c]
        color = STYLE['cluster_colors'][c % len(STYLE['cluster_colors'])]
        subset = hourly_by_cluster[hourly_by_cluster['PU_cluster'] == cluster_idx]
        fig_comp.add_trace(go.Bar(
            x=subset['hour'], y=subset['share'],
            name=label, marker_color=color,
            legendgroup=str(c), showlegend=(col_idx == 1),
            hovertemplate=f'{label}<br>Hour %{{x}}:00: %{{y:.1f}}%<extra></extra>',
        ), row=1, col=col_idx)

fig_comp.update_layout(**base_layout(),
    barmode='stack',
    legend=dict(x=0.85, y=0.98, font=dict(family=STYLE['font_family'])))
fig_comp.update_xaxes(title_text='Hour of Day')
fig_comp.update_yaxes(title_text='Cluster Share (%)', col=1)
save_html(fig_comp, 'fix_hourly_composition.html')


# %% [markdown]
# ---
# ## 13. Lorenz Curve + Gini

# %%
print("Lorenz curve...")
fig = go.Figure()
fig.add_trace(go.Scatter(x=cz_18, y=ct_18, mode='lines', name=f'2018 (Gini={gini_18:.3f})',
    line=dict(color=STYLE['year_2018'], width=2.5)))
fig.add_trace(go.Scatter(x=cz_25, y=ct_25, mode='lines', name=f'2025 (Gini={gini_25:.3f})',
    line=dict(color=STYLE['year_2025'], width=2.5)))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Perfect equality',
    line=dict(color='#888', dash='dash', width=1)))
fig.update_layout(**base_layout(),
    xaxis=styled_axis(title_text='Cumulative Share of Zones'),
    yaxis=styled_axis(title_text='Cumulative Share of Trips'),
    legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])))
save_html(fig, 'lorenz_curve.html')


# %% [markdown]
# ---
# ## 14. LISA Maps (2018 + 2025)

# %%
print("LISA cluster maps...")
for year_label, df_year in [('2018', df_2018), ('2025', df_2025)]:
    zone_counts = df_year.groupby('PU_zone_id').size()
    gdf_tmp = gdf_base.copy()
    gdf_tmp['trips'] = gdf_tmp['LocationID_int'].map(zone_counts).fillna(0)
    gdf_tmp['trip_share'] = 100 * gdf_tmp['trips'] / gdf_tmp['trips'].sum()

    lisa = Moran_Local(gdf_tmp['trip_share'].values, w, permutations=999)
    mi_global = Moran(gdf_tmp['trip_share'].values, w)

    sig = lisa.p_sim < 0.05
    quadrant_map = {1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}
    gdf_tmp['lisa_class'] = 'ns'
    for idx in range(len(gdf_tmp)):
        if sig[idx]:
            gdf_tmp.iloc[idx, gdf_tmp.columns.get_loc('lisa_class')] = \
                quadrant_map.get(lisa.q[idx], 'ns')

    gdf_tmp['zone_id_str'] = gdf_tmp['LocationID_int'].astype(str)
    gdf_tmp['zone_name'] = gdf_tmp['LocationID_int'].map(zc_lookup['zone_name'])
    gdf_tmp['borough_name'] = gdf_tmp['LocationID_int'].map(zc_lookup['borough'])

    geojson_all = json.loads(gdf_tmp.to_json())
    for feat, zid, lclass in zip(geojson_all['features'], gdf_tmp['zone_id_str'], gdf_tmp['lisa_class']):
        feat['properties']['zone_id_str'] = zid
        feat['properties']['lisa_class'] = lclass

    fig_lisa = go.Figure()
    fig_lisa.add_trace(go.Choroplethmap(
        geojson=taxi_zones_geo_4326, locations=all_zone_ids,
        featureidkey='properties.LocationID',
        z=[1] * len(all_zone_ids), colorscale=[[0, '#f5f5f5'], [1, '#f5f5f5']],
        marker_opacity=0.3, marker_line_width=0.3, marker_line_color='#ccc',
        showscale=False, hoverinfo='skip'))

    for lclass in ['HH', 'LL', 'HL', 'LH', 'ns']:
        subset = gdf_tmp[gdf_tmp['lisa_class'] == lclass]
        if len(subset) == 0:
            continue
        subset_geojson = {
            'type': 'FeatureCollection',
            'features': [f for f in geojson_all['features']
                         if f['properties']['lisa_class'] == lclass]
        }
        fig_lisa.add_trace(go.Choroplethmap(
            geojson=subset_geojson,
            locations=subset['zone_id_str'].values,
            featureidkey='properties.zone_id_str',
            z=[1] * len(subset),
            colorscale=[[0, STYLE['lisa_colors'][lclass]], [1, STYLE['lisa_colors'][lclass]]],
            marker_opacity=0.8 if lclass != 'ns' else 0.35,
            marker_line_width=0.5, marker_line_color='rgba(255,255,255,0.6)',
            showscale=False, name=STYLE['lisa_labels'][lclass], showlegend=True,
            customdata=np.column_stack([
                subset['zone_name'].fillna('Unknown').values,
                subset['borough_name'].fillna('Unknown').values,
                subset['trip_share'].values]),
            hovertemplate=(
                '<b>%{customdata[0]}</b> (%{customdata[1]})<br>'
                'Trip share: %{customdata[2]:.2f}%<br>'
                f'LISA: {STYLE["lisa_labels"][lclass]}<extra></extra>')))

    fig_lisa.update_layout(
        **base_layout(height=700, width=1100, margin=STYLE['margin_map']),
        map_style=STYLE['map_style'], map_zoom=STYLE['map_zoom'],
        map_center=dict(**STYLE['map_center']),
        legend=dict(
            title="LISA Classification", yanchor='top', y=0.99, xanchor='left', x=0.01,
            bgcolor='rgba(255,255,255,0.95)', bordercolor='#dde1e7', borderwidth=1,
            font=dict(size=STYLE['legend_size'], family=STYLE['font_family'])),
        annotations=[dict(
            x=0.99, y=0.01, xref='paper', yref='paper', xanchor='right', yanchor='bottom',
            text=f"Global Moran's I: {mi_global.I:.4f} (p={mi_global.p_sim:.4f})",
            showarrow=False, font=dict(size=11, family=STYLE['font_family']),
            bgcolor='rgba(255,255,255,0.9)', bordercolor='#ddd', borderwidth=1)])
    save_html(fig_lisa, f'lisa_map_{year_label}.html')


# %% [markdown]
# ---
# ## 15. Market Share Breakdown (ext1)

# %%
print("Market share breakdown...")
# Reload just the filter columns to count all operators
table_ms_2018 = pq.read_table(PATH_2018, columns=['dispatching_base_num'])
df_ms_2018 = table_ms_2018.to_pandas()
n_uber_2018 = df_ms_2018['dispatching_base_num'].isin(UBER_2018_BASES).sum()
n_lyft_2018 = df_ms_2018['dispatching_base_num'].isin(LYFT_2018_BASES).sum()
n_other_2018 = len(df_ms_2018) - n_uber_2018 - n_lyft_2018
total_ms_2018 = len(df_ms_2018)
del table_ms_2018, df_ms_2018; gc.collect()

table_ms_2025 = pq.read_table(PATH_2025, columns=['hvfhs_license_num'])
df_ms_2025 = table_ms_2025.to_pandas()
n_uber_2025_ms = (df_ms_2025['hvfhs_license_num'] == UBER_2025_LICENSE).sum()
n_lyft_2025 = (df_ms_2025['hvfhs_license_num'] == LYFT_2025_LICENSE).sum()
n_other_2025 = len(df_ms_2025) - n_uber_2025_ms - n_lyft_2025
total_ms_2025 = len(df_ms_2025)
del table_ms_2025, df_ms_2025; gc.collect()

ms_data = pd.DataFrame({
    'Operator': ['Uber', 'Lyft', 'Other FHV'] * 2,
    'Year': ['2018'] * 3 + ['2025'] * 3,
    'Trips': [n_uber_2018, n_lyft_2018, n_other_2018,
              n_uber_2025_ms, n_lyft_2025, n_other_2025],
    'Share': [100 * n_uber_2018 / total_ms_2018, 100 * n_lyft_2018 / total_ms_2018,
              100 * n_other_2018 / total_ms_2018, 100 * n_uber_2025_ms / total_ms_2025,
              100 * n_lyft_2025 / total_ms_2025, 100 * n_other_2025 / total_ms_2025]
})

fig_ms = go.Figure()
for op in ['Uber', 'Lyft', 'Other FHV']:
    subset = ms_data[ms_data['Operator'] == op]
    fig_ms.add_trace(go.Bar(
        x=subset['Year'], y=subset['Share'], name=op, marker_color=OPERATOR_COLORS[op],
        text=[f"{s:.1f}%" for s in subset['Share']], textposition='outside',
        customdata=subset['Trips'].values,
        hovertemplate=(f'<b>{op}</b><br>Year: %{{x}}<br>Share: %{{y:.1f}}%<br>'
                       f'Trips: %{{customdata:,}}<extra></extra>')))
fig_ms.update_layout(**base_layout(),
    xaxis=styled_axis(title_text=''), yaxis=styled_axis(title_text='Market Share (%)', range=[0, 85]),
    barmode='group', legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])))
save_html(fig_ms, 'ext1_market_share_breakdown.html')

# Stacked version
fig_ms_stack = make_subplots(rows=1, cols=2,
    subplot_titles=('Absolute Trip Counts', 'Market Share (%)'), horizontal_spacing=0.12)
for op in ['Uber', 'Lyft', 'Other FHV']:
    subset = ms_data[ms_data['Operator'] == op]
    fig_ms_stack.add_trace(go.Bar(x=subset['Year'], y=subset['Trips'], name=op,
        marker_color=OPERATOR_COLORS[op], legendgroup=op, showlegend=True,
        hovertemplate=f'<b>{op}</b><br>Trips: %{{y:,}}<extra></extra>'), row=1, col=1)
    fig_ms_stack.add_trace(go.Bar(x=subset['Year'], y=subset['Share'], name=op,
        marker_color=OPERATOR_COLORS[op], legendgroup=op, showlegend=False,
        hovertemplate=f'<b>{op}</b><br>Share: %{{y:.1f}}%<extra></extra>'), row=1, col=2)
fig_ms_stack.update_layout(**base_layout(), barmode='stack',
    legend=dict(x=0.85, y=0.98, font=dict(family=STYLE['font_family'])))
fig_ms_stack.update_yaxes(title_text='Total Trips', col=1)
fig_ms_stack.update_yaxes(title_text='Share (%)', col=2)
save_html(fig_ms_stack, 'ext1_market_share_stacked.html')


# %% [markdown]
# ---
# ## 16. OD Flow Sankey + Heatmaps (ext3)

# %%
print("OD flow Sankeys...")
for year_label, df_year, c_names in [('2018', df_2018, cnames_2018), ('2025', df_2025, cnames_2025)]:
    valid = df_year.dropna(subset=['PU_cluster', 'DO_cluster']).copy()
    valid['PU_cluster'] = valid['PU_cluster'].astype(int)
    valid['DO_cluster'] = valid['DO_cluster'].astype(int)
    od = valid.groupby(['PU_cluster', 'DO_cluster']).size().reset_index(name='trips')

    labels_pu = [f"PU: {short_label(c_names[i])}" for i in range(N_CLUSTERS)]
    labels_do = [f"DO: {short_label(c_names[i])}" for i in range(N_CLUSTERS)]
    node_colors = [STYLE['cluster_colors'][i % len(STYLE['cluster_colors'])] for i in range(N_CLUSTERS)] * 2
    sources = od['PU_cluster'].astype(int).tolist()
    targets = (od['DO_cluster'].astype(int) + N_CLUSTERS).tolist()
    link_colors = [hex_to_rgba(STYLE['cluster_colors'][s % len(STYLE['cluster_colors'])], 0.3) for s in sources]

    fig_sankey = go.Figure(go.Sankey(
        node=dict(label=labels_pu + labels_do, color=node_colors, pad=15, thickness=20),
        link=dict(source=sources, target=targets, value=od['trips'].tolist(), color=link_colors)))
    fig_sankey.update_layout(**base_layout(height=550, width=1000, margin=dict(l=20, r=20, t=30, b=20)))
    save_html(fig_sankey, f'ext3_od_sankey_{year_label}.html')

# OD change heatmap
print("OD change heatmap...")
cl_short = [short_labels_2018[i] for i in range(N_CLUSTERS)]
fig = go.Figure(go.Heatmap(
    z=od_diff, x=cl_short, y=cl_short,
    colorscale='RdBu', zmid=0,
    colorbar=dict(title=dict(text='Change (pp)'), ticksuffix=' pp',
                  tickfont=dict(family=STYLE['font_family'])),
    hovertemplate='PU: %{y}<br>DO: %{x}<br>Change: %{z:.2f} pp<extra></extra>',
    text=np.round(od_diff, 2), texttemplate='%{text:.1f}', textfont=dict(size=10)))
fig.update_layout(**base_layout(),
    xaxis=styled_axis(title_text='Dropoff Cluster'),
    yaxis=styled_axis(title_text='Pickup Cluster'))
save_html(fig, 'fix_od_change.html')


# %% [markdown]
# ---
# ## 17. Dropoff Density Change Map (ext3)

# %%
print("Dropoff density change map...")
fig_do = go.Figure()
add_bg_layer(fig_do)
zids_do = set(do_comparison['zone_id'].values)
fgeo_do = filter_geojson(zids_do)
fig_do.add_trace(go.Choroplethmap(
    geojson=fgeo_do, locations=do_comparison['zone_id'],
    featureidkey='properties.LocationID', z=do_comparison['share_change_capped'],
    colorscale='RdBu', zmid=0, marker_opacity=0.85, marker_line_width=0.5,
    marker_line_color='rgba(255,255,255,0.6)',
    colorbar=dict(title=dict(text='Change (pp)', font=dict(family=STYLE['font_family'])),
                  ticksuffix=' pp', x=0.99, tickfont=dict(family=STYLE['font_family'])),
    customdata=np.column_stack([
        do_comparison['zone_name'].fillna('Unknown'), do_comparison['borough'].fillna('Unknown'),
        do_comparison['share_change']]),
    hovertemplate='<b>%{customdata[0]}</b> (%{customdata[1]})<br>DO Change: %{customdata[2]:.2f} pp<extra></extra>'))
fig_do.update_layout(**base_layout(height=700, width=1100, margin=STYLE['margin_map']),
    map_style=STYLE['map_style'], map_zoom=STYLE['map_zoom'], map_center=dict(**STYLE['map_center']))
save_html(fig_do, 'ext3_dropoff_density_change.html')


# %% [markdown]
# ---
# ## 18. Trip Distance Distribution + By Cluster (ext4 / fix)

# %%
print("Trip distance distribution...")
fig = go.Figure()
for label, data, color in [('2018', d18_valid, STYLE['year_2018']), ('2025', d25_valid, STYLE['year_2025'])]:
    fig.add_trace(go.Histogram(x=data, histnorm='probability density', nbinsx=80, name=label,
        marker_color=color, opacity=0.6,
        hovertemplate='%{x:.1f} km<br>Density: %{y:.4f}<extra>' + label + '</extra>'))
fig.update_layout(**base_layout(),
    xaxis=styled_axis(title_text='Trip Distance (km)', range=[0, 30]),
    yaxis=styled_axis(title_text='Density'), barmode='overlay',
    legend=dict(x=0.75, y=0.95, font=dict(family=STYLE['font_family'])),
    annotations=[dict(
        x=0.98, y=0.95, xref='paper', yref='paper', xanchor='right',
        text=(f"Median: {d18_valid.median():.2f} km (2018) vs {d25_valid.median():.2f} km (2025)<br>"
              f"Mean: {d18_valid.mean():.2f} km vs {d25_valid.mean():.2f} km"),
        showarrow=False, font=dict(size=11, family=STYLE['font_family']),
        bgcolor='rgba(255,255,255,0.9)', bordercolor='#ddd', borderwidth=1)])
save_html(fig, 'fix_trip_distance.html')

# Distance by cluster
print("Distance by cluster...")
dist_data = []
for i in range(N_CLUSTERS):
    d18c = df_2018[(df_2018['PU_cluster'] == i) & (df_2018['trip_dist_km'] > 0) & (df_2018['trip_dist_km'] < 100)]['trip_dist_km']
    i25 = cluster_matches[i]
    d25c = df_2025[(df_2025['PU_cluster'] == i25) & (df_2025['trip_dist_km'] > 0) & (df_2025['trip_dist_km'] < 100)]['trip_dist_km']
    dist_data.append({
        'cluster': short_labels_2018[i],
        'median_2018': d18c.median() if len(d18c) > 0 else 0,
        'median_2025': d25c.median() if len(d25c) > 0 else 0})
ddf = pd.DataFrame(dist_data)
fig = go.Figure()
fig.add_trace(go.Bar(x=ddf['cluster'], y=ddf['median_2018'], name='2018 (median)',
    marker_color=STYLE['year_2018'],
    text=[f"{v:.1f}" for v in ddf['median_2018']], textposition='outside',
    hovertemplate='%{x}<br>2018 median: %{y:.2f} km<extra></extra>'))
fig.add_trace(go.Bar(x=ddf['cluster'], y=ddf['median_2025'], name='2025 (median)',
    marker_color=STYLE['year_2025'],
    text=[f"{v:.1f}" for v in ddf['median_2025']], textposition='outside',
    hovertemplate='%{x}<br>2025 median: %{y:.2f} km<extra></extra>'))
fig.update_layout(**base_layout(),
    xaxis=styled_axis(title_text='Pickup Cluster'),
    yaxis=styled_axis(title_text='Median Trip Distance (km)'),
    barmode='group', legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])))
save_html(fig, 'fix_distance_by_cluster.html')


# %% [markdown]
# ---
# ## 19. JSD by Cluster (ext7 / fix)

# %%
print("JSD by cluster...")
fig = go.Figure()
fig.add_trace(go.Bar(
    x=cjsd_df['cluster'], y=cjsd_df['jsd'],
    marker_color=[STYLE['cluster_colors'][r['idx'] % len(STYLE['cluster_colors'])] for _, r in cjsd_df.iterrows()],
    hovertemplate='%{x}<br>JSD: %{y:.5f}<br><i>Higher = more temporal change</i><extra></extra>'))
fig.add_hline(y=agg_jsd, line_dash='dash', line_color='#333', line_width=2,
    annotation_text=f'Citywide average: {agg_jsd:.4f}', annotation_position='top right')
fig.update_layout(**base_layout(),
    xaxis=styled_axis(title_text='Geographic Cluster'),
    yaxis=styled_axis(title_text='Temporal Shift (JSD)', tickformat='.4f'),
    showlegend=False,
    annotations=[dict(
        x=0.5, y=-0.18, xref='paper', yref='paper',
        text="JSD measures how much the hourly demand profile changed between 2018 and 2025. Values near 0 = no change.",
        showarrow=False, font=dict(size=10, color='#777', family=STYLE['font_family']))])
save_html(fig, 'fix_jsd_by_cluster.html')


# %% [markdown]
# ---
# ## 20. Localization Distribution + Map (ext6 / fix)

# %%
print("Localization distribution...")
fig = go.Figure()
for label, data, color in [('2018', local_2018, STYLE['year_2018']), ('2025', local_2025, STYLE['year_2025'])]:
    fig.add_trace(go.Histogram(x=data['local_share'], histnorm='percent', nbinsx=40,
        name=label, marker_color=color, opacity=0.6,
        hovertemplate='%{x:.0f}%: %{y:.1f}% of zones<extra>' + label + '</extra>'))
fig.add_vline(x=local_2018['local_share'].mean(), line_dash='dash', line_color=STYLE['year_2018'], line_width=2,
    annotation_text=f"2018 mean: {mean_local_2018:.1f}%", annotation_position='top left')
fig.add_vline(x=local_2025['local_share'].mean(), line_dash='dash', line_color=STYLE['year_2025'], line_width=2,
    annotation_text=f"2025 mean: {mean_local_2025:.1f}%", annotation_position='top right')
fig.update_layout(**base_layout(),
    xaxis=styled_axis(title_text=f'% of Trips to Same or {K_NEAREST}-Nearest Zones'),
    yaxis=styled_axis(title_text='% of Zones'), barmode='overlay',
    legend=dict(x=0.75, y=0.95, font=dict(family=STYLE['font_family'])))
save_html(fig, 'fix_localization_pct.html')

# Localization change map
print("Localization change map...")
local_merged = local_2018[['local_share']].rename(columns={'local_share': 'local_2018'}).join(
    local_2025[['local_share']].rename(columns={'local_share': 'local_2025'}), how='outer').fillna(0)
local_merged['change'] = local_merged['local_2025'] - local_merged['local_2018']
local_merged = local_merged.reset_index()
local_merged.columns = ['zone_id', 'local_2018', 'local_2025', 'change']
local_merged['zone_id'] = local_merged['zone_id'].astype(str)
cap_local = local_merged['change'].abs().quantile(0.95)
local_merged['change_capped'] = local_merged['change'].clip(-cap_local, cap_local)
local_merged['zone_name'] = local_merged['zone_id'].astype(int).map(zc_lookup['zone_name'])
local_merged['borough'] = local_merged['zone_id'].astype(int).map(zc_lookup['borough'])

fig_lm = go.Figure()
add_bg_layer(fig_lm)
zids_local = set(local_merged['zone_id'].values)
fgeo_local = filter_geojson(zids_local)
fig_lm.add_trace(go.Choroplethmap(
    geojson=fgeo_local, locations=local_merged['zone_id'],
    featureidkey='properties.LocationID', z=local_merged['change_capped'],
    colorscale='RdBu', zmid=0, marker_opacity=0.85, marker_line_width=0.5,
    marker_line_color='rgba(255,255,255,0.6)',
    colorbar=dict(title=dict(text='Change (pp)', font=dict(family=STYLE['font_family'])),
                  ticksuffix=' pp', x=0.99, tickfont=dict(family=STYLE['font_family'])),
    customdata=np.column_stack([
        local_merged['zone_name'].fillna('Unknown'), local_merged['borough'].fillna('Unknown'),
        local_merged['change'], local_merged['local_2018'], local_merged['local_2025']]),
    hovertemplate=('<b>%{customdata[0]}</b> (%{customdata[1]})<br>'
        'Local share 2018: %{customdata[3]:.1f}%<br>Local share 2025: %{customdata[4]:.1f}%<br>'
        'Change: %{customdata[2]:.1f} pp<extra></extra>')))
fig_lm.update_layout(**base_layout(height=700, width=1100, margin=STYLE['margin_map']),
    map_style=STYLE['map_style'], map_zoom=STYLE['map_zoom'], map_center=dict(**STYLE['map_center']))
save_html(fig_lm, 'ext6_localization_change_map.html')


# %% [markdown]
# ---
# ## 21. Bootstrap Gini (ext9) + Mismatch Ratios (ext10)

# %%
print("Bootstrap Gini plot...")
fig_boot = go.Figure()
fig_boot.add_trace(go.Histogram(x=gini_boot_2018, nbinsx=50, name='2018',
    marker_color=STYLE['year_2018'], opacity=0.6))
fig_boot.add_trace(go.Histogram(x=gini_boot_2025, nbinsx=50, name='2025',
    marker_color=STYLE['year_2025'], opacity=0.6))
for ci, color in [(ci_2018, STYLE['year_2018']), (ci_2025, STYLE['year_2025'])]:
    for bound in ci:
        fig_boot.add_vline(x=bound, line_dash='dot', line_color=color, line_width=1.5)
fig_boot.update_layout(**base_layout(),
    xaxis=styled_axis(title_text='Gini Coefficient'),
    yaxis=styled_axis(title_text='Bootstrap Count'), barmode='overlay',
    legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])),
    annotations=[dict(x=0.99, y=0.99, xref='paper', yref='paper', xanchor='right', yanchor='top',
        text=(f"2018: {gini_18:.4f} [{ci_2018[0]:.4f}, {ci_2018[1]:.4f}]<br>"
              f"2025: {gini_25:.4f} [{ci_2025[0]:.4f}, {ci_2025[1]:.4f}]"),
        showarrow=False, font=dict(size=11, family=STYLE['font_family']),
        bgcolor='rgba(255,255,255,0.9)', bordercolor='#ddd', borderwidth=1)])
save_html(fig_boot, 'ext9_bootstrap_gini.html')

# Mismatch distributions
print("Mismatch ratio distributions...")
fig_mm = go.Figure()
fig_mm.add_trace(go.Histogram(x=mr_2018['ratio'].dropna(), nbinsx=50, name='2018',
    marker_color=STYLE['year_2018'], opacity=0.6))
fig_mm.add_trace(go.Histogram(x=mr_2025['ratio'].dropna(), nbinsx=50, name='2025',
    marker_color=STYLE['year_2025'], opacity=0.6))
fig_mm.add_vline(x=0.5, line_dash='dash', line_color='#333', line_width=1.5,
    annotation_text='Perfect PU/DO balance', annotation_position='top left')
fig_mm.update_layout(**base_layout(),
    xaxis=styled_axis(title_text='PU / (PU + DO) Ratio'),
    yaxis=styled_axis(title_text='Number of Zones'), barmode='overlay',
    legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])),
    annotations=[dict(x=0.99, y=0.99, xref='paper', yref='paper', xanchor='right', yanchor='top',
        text=(f"KS: D={ks_stat:.4f}, p={ks_p:.2e}<br>Levene: W={lev_stat:.4f}, p={lev_p:.2e}"),
        showarrow=False, font=dict(size=10, family=STYLE['font_family']),
        bgcolor='rgba(255,255,255,0.9)', bordercolor='#ddd', borderwidth=1)])
save_html(fig_mm, 'ext10_mismatch_ratios.html')

# Mismatch change map
print("Mismatch change map...")
mr_merged = mr_2018[['ratio']].rename(columns={'ratio': 'ratio_2018'}).join(
    mr_2025[['ratio']].rename(columns={'ratio': 'ratio_2025'}), how='outer')
mr_merged['ratio_change'] = mr_merged['ratio_2025'] - mr_merged['ratio_2018']
mr_merged = mr_merged.dropna(subset=['ratio_change']).reset_index()
mr_merged.columns = ['zone_id', 'ratio_2018', 'ratio_2025', 'ratio_change']
mr_merged['zone_id'] = mr_merged['zone_id'].astype(float).astype(int).astype(str)
mr_merged['zone_name'] = mr_merged['zone_id'].astype(int).map(zc_lookup['zone_name'])
mr_merged['borough'] = mr_merged['zone_id'].astype(int).map(zc_lookup['borough'])
cap_mm = mr_merged['ratio_change'].abs().quantile(0.95)
mr_merged['ratio_change_capped'] = mr_merged['ratio_change'].clip(-cap_mm, cap_mm)

fig_mm_map = go.Figure()
add_bg_layer(fig_mm_map)
zids_mm = set(mr_merged['zone_id'].values)
fig_mm_map.add_trace(go.Choroplethmap(
    geojson=filter_geojson(zids_mm), locations=mr_merged['zone_id'],
    featureidkey='properties.LocationID', z=mr_merged['ratio_change_capped'],
    colorscale='RdBu', zmid=0, marker_opacity=0.85, marker_line_width=0.5,
    marker_line_color='rgba(255,255,255,0.6)',
    colorbar=dict(title=dict(text='Change in PU/(PU+DO)', font=dict(family=STYLE['font_family'])),
                  x=0.99, tickfont=dict(family=STYLE['font_family'])),
    customdata=np.column_stack([mr_merged['zone_name'].fillna('Unknown'),
        mr_merged['borough'].fillna('Unknown'),
        mr_merged['ratio_2018'], mr_merged['ratio_2025'], mr_merged['ratio_change']]),
    hovertemplate=('<b>%{customdata[0]}</b> (%{customdata[1]})<br>'
        '2018: %{customdata[2]:.3f}<br>2025: %{customdata[3]:.3f}<br>'
        'Change: %{customdata[4]:.3f}<extra></extra>')))
fig_mm_map.update_layout(**base_layout(height=700, width=1100, margin=STYLE['margin_map']),
    map_style=STYLE['map_style'], map_zoom=STYLE['map_zoom'], map_center=dict(**STYLE['map_center']))
save_html(fig_mm_map, 'ext10_mismatch_change_map.html')


# %% [markdown]
# ---
# ## 22. Demand Profiles + Maps (ext5 / fix)

# %%
print("Demand profiles and maps...")
for year_label, df_year in [('2018', df_2018), ('2025', df_2025)]:
    hourly = df_year.groupby(['PU_zone_id', 'hour']).size().reset_index(name='trips')
    pivot = hourly.pivot(index='PU_zone_id', columns='hour', values='trips').fillna(0)
    pivot_norm = pivot.div(pivot.sum(axis=1), axis=0).fillna(0)

    zone_totals = df_year.groupby('PU_zone_id').size()
    valid = zone_totals[zone_totals >= 50].index
    pivot_norm = pivot_norm.loc[pivot_norm.index.isin(valid)]

    X = StandardScaler().fit_transform(pivot_norm.values)
    km = KMeans(n_clusters=N_DEMAND_CLUSTERS, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    pivot_norm['dc'] = labels

    dc_names = {}
    for c in range(N_DEMAND_CLUSTERS):
        mask = pivot_norm['dc'] == c
        avg = pivot_norm.loc[mask].drop(columns='dc').mean()
        peak = avg.idxmax()
        n = mask.sum()
        if peak <= 9: pat = 'Morning-peak'
        elif peak <= 14: pat = 'Midday'
        elif peak <= 19: pat = 'Evening-peak'
        else: pat = 'Night-peak'
        dc_names[c] = f"{pat} ({n} zones, peak ~{peak}:00)"

    # Profile line chart
    fig = go.Figure()
    for c in range(N_DEMAND_CLUSTERS):
        mask = pivot_norm['dc'] == c
        avg = pivot_norm.loc[mask].drop(columns=['dc']).mean()
        fig.add_trace(go.Scatter(
            x=list(range(24)), y=avg.values, mode='lines+markers', name=dc_names[c],
            line=dict(color=DEMAND_COLORS[c % len(DEMAND_COLORS)], width=2.5),
            marker=dict(size=5),
            hovertemplate='Hour %{x}:00<br>Share: %{y:.3f}<extra>' + dc_names[c] + '</extra>'))
    fig.update_layout(**base_layout(),
        xaxis=styled_axis(title_text='Hour of Day', dtick=2),
        yaxis=styled_axis(title_text='Within-Zone Trip Share'),
        legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])),
        annotations=[dict(x=0.5, y=-0.18, xref='paper', yref='paper',
            text=f"Zones grouped by when their demand peaks. Each line is the average hourly profile ({year_label}).",
            showarrow=False, font=dict(size=10, color='#777', family=STYLE['font_family']))])
    save_html(fig, f'fix_demand_profiles_{year_label}.html')

    # Demand type map
    zone_demand = pivot_norm[['dc']].reset_index()
    zone_demand.columns = ['zone_id', 'dc']
    zone_demand['dc_name'] = zone_demand['dc'].map(dc_names)
    zone_demand['zone_id'] = zone_demand['zone_id'].astype(float).astype(int).astype(str)
    zone_demand['zone_name'] = zone_demand['zone_id'].astype(int).map(zc_lookup['zone_name'])
    zone_demand['borough'] = zone_demand['zone_id'].astype(int).map(zc_lookup['borough'])

    zids = set(zone_demand['zone_id'].values)
    fgeo = filter_geojson(zids)

    fig_map = go.Figure()
    add_bg_layer(fig_map)
    for c in range(N_DEMAND_CLUSTERS):
        subset = zone_demand[zone_demand['dc'] == c]
        if len(subset) == 0: continue
        sgeo = {'type': 'FeatureCollection',
                'features': [f for f in fgeo['features']
                             if f['properties']['LocationID'] in set(subset['zone_id'].values)]}
        fig_map.add_trace(go.Choroplethmap(
            geojson=sgeo, locations=subset['zone_id'].values,
            featureidkey='properties.LocationID',
            z=[1] * len(subset), colorscale=[[0, DEMAND_COLORS[c]], [1, DEMAND_COLORS[c]]],
            marker_opacity=0.8, marker_line_width=0.5,
            marker_line_color='rgba(255,255,255,0.7)',
            showscale=False, name=dc_names[c], showlegend=True,
            customdata=np.column_stack([
                subset['zone_name'].fillna('Unknown').values,
                subset['borough'].fillna('Unknown').values,
                subset['dc_name'].values]),
            hovertemplate='<b>%{customdata[0]}</b> (%{customdata[1]})<br>Type: %{customdata[2]}<extra></extra>'))
    fig_map.update_layout(
        **base_layout(height=650, width=1100, margin=STYLE['margin_map']),
        legend=dict(title="Demand Type", yanchor="top", y=0.99, xanchor="left", x=0.01,
            bgcolor="rgba(255,255,255,0.95)", bordercolor="#dde1e7", borderwidth=1,
            font=dict(size=10, family=STYLE['font_family'])),
        map=dict(style=STYLE['map_style'], center=STYLE['map_center'], zoom=STYLE['map_zoom']))
    save_html(fig_map, f'fix_demand_map_{year_label}.html')


# %% [markdown]
# ---
# ## 23. Graduated Prediction Scorecard (ext8)

# %%
print("Graduated scorecard...")
scorecard = [
    {'prediction': 'Increased airport demand',
     'result': 'JFK + LaGuardia combined pickup share rose 48%',
     'strength': 'Strong', 'strength_color': '#2d6a4f',
     'notes': 'Clear directional change with large magnitude. Airport ground transportation policies also changed over this period.'},
    {'prediction': 'Manhattan core decline',
     'result': 'Top-3 zones lost 34-39% of their pickup share',
     'strength': 'Strong', 'strength_color': '#2d6a4f',
     'notes': 'Large, consistent decline across multiple central zones. Post-pandemic remote work is an equally plausible explanation.'},
    {'prediction': 'Lower geographic concentration',
     'result': f'Gini: {gini_18:.3f} to {gini_25:.3f} (delta = {gini_25 - gini_18:.3f})',
     'strength': 'Moderate', 'strength_color': '#b5830b',
     'notes': 'Direction is correct and magnitude is non-trivial. Without a counterfactual, unclear whether this exceeds what market share growth alone would produce.'},
    {'prediction': 'Reduced spatial clustering',
     'result': f"Moran's I: {mi_2018:.3f} to {mi_2025:.3f}",
     'strength': 'Strong', 'strength_color': '#2d6a4f',
     'notes': "Nearly halved. Both values significant at p<0.001. Most robust finding because Moran's I is less mechanically affected by volume growth than Gini."},
    {'prediction': 'More localised matching',
     'result': f'Intra-cluster trips: {intra_2018_pct:.1f}% to {intra_2025_pct:.1f}%',
     'strength': 'Weak', 'strength_color': '#c1121f',
     'notes': f'Clusters are fit independently per year, so boundary changes could produce this result. Zone-level localization (K={K_NEAREST}) shows {mean_local_2018:.1f}% to {mean_local_2025:.1f}%, modest.'},
    {'prediction': 'Stable geographic structure',
     'result': f'Mean centroid shift: {avg_shift:.2f} km; temporal profiles stable (JSD={agg_jsd:.4f})',
     'strength': 'Moderate', 'strength_color': '#b5830b',
     'notes': 'Centroid shifts are small relative to NYC extent (~30 km). But one cluster shifted 11.6 km. Temporal JSD is low, supporting the argument.'},
]

scorecard_html = """<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8"><style>
body { font-family: 'Public Sans', -apple-system, sans-serif; max-width: 1000px; margin: 2rem auto; padding: 0 1.5rem; color: #1a2744; }
table { width: 100%; border-collapse: collapse; font-size: 0.95rem; }
th { text-align: left; padding: 0.75rem 1rem; background: #f8f9fa; border-bottom: 2px solid #dde1e7; }
td { padding: 0.75rem 1rem; border-bottom: 1px solid #eee; vertical-align: top; }
.strength { font-weight: 600; padding: 0.2rem 0.6rem; border-radius: 3px; color: white; display: inline-block; font-size: 0.85rem; }
.notes { font-size: 0.88rem; color: #555; line-height: 1.5; }
</style></head><body>
<h2>Prediction Scorecard (Graduated)</h2>
<table><thead><tr><th>Prediction</th><th>Result</th><th>Strength</th><th>Notes</th></tr></thead><tbody>
"""
for s in scorecard:
    scorecard_html += f"""<tr><td><strong>{s['prediction']}</strong></td>
<td>{s['result']}</td>
<td><span class="strength" style="background:{s['strength_color']};">{s['strength']}</span></td>
<td class="notes">{s['notes']}</td></tr>\n"""
scorecard_html += "</tbody></table></body></html>"
with open(OUTPUT_DIR + 'ext8_graduated_scorecard.html', 'w') as f:
    f.write(scorecard_html)
print("  Saved: ext8_graduated_scorecard.html")


# %% [markdown]
# ---
# ## 24. Multi-Year Time Series (ext2, if intermediate files exist)

# %%
print("Multi-year time series (scanning for intermediate files)...")
year_configs = {
    2018: {'path': DATA_DIR + 'fhv_tripdata_2018-01.parquet', 'uber_col': 'dispatching_base_num', 'uber_val': UBER_2018_BASES, 'pu_col': 'PUlocationID'},
    2019: {'path': DATA_DIR + 'fhv_tripdata_2019-01.parquet', 'uber_col': 'dispatching_base_num', 'uber_val': UBER_2018_BASES, 'pu_col': 'PUlocationID'},
}
for yr in range(2020, 2026):
    year_configs[yr] = {'path': DATA_DIR + f'fhvhv_tripdata_{yr}-01.parquet',
        'uber_col': 'hvfhs_license_num', 'uber_val': UBER_2025_LICENSE, 'pu_col': 'PULocationID'}

ts_results = []
for year, cfg in sorted(year_configs.items()):
    if not os.path.exists(cfg['path']):
        print(f"  [SKIP] {year}: file not found")
        continue
    print(f"  [Processing {year}]...")
    try:
        cols = [cfg['pu_col'], cfg['uber_col']]
        table = pq.read_table(cfg['path'], columns=cols)
        df_tmp = table.to_pandas()
        total = len(df_tmp)
        if isinstance(cfg['uber_val'], list):
            df_tmp = df_tmp[df_tmp[cfg['uber_col']].isin(cfg['uber_val'])].copy()
        else:
            df_tmp = df_tmp[df_tmp[cfg['uber_col']] == cfg['uber_val']].copy()
        uber_share = 100 * len(df_tmp) / total
        df_tmp = df_tmp.dropna(subset=[cfg['pu_col']])
        df_tmp[cfg['pu_col']] = df_tmp[cfg['pu_col']].astype(int)
        zc_tmp = df_tmp.groupby(cfg['pu_col']).size()
        cz, ct = lorenz_data(zc_tmp)
        gini = gini_from_lorenz(cz, ct)
        gdf_tmp = gdf_base.copy()
        gdf_tmp['trips'] = gdf_tmp['LocationID_int'].map(zc_tmp).fillna(0)
        gdf_tmp['trip_share'] = 100 * gdf_tmp['trips'] / gdf_tmp['trips'].sum()
        mi = Moran(gdf_tmp['trip_share'].values, w)
        ts_results.append({'year': year, 'uber_share': uber_share, 'gini': gini, 'morans_i': mi.I})
        print(f"    Uber share: {uber_share:.1f}%, Gini: {gini:.3f}, Moran's I: {mi.I:.3f}")
        del table, df_tmp, gdf_tmp; gc.collect()
    except Exception as e:
        print(f"    ERROR: {e}")

ts_df = pd.DataFrame(ts_results)
if len(ts_df) >= 2:
    fig_ts = make_subplots(rows=1, cols=2,
        subplot_titles=("Gini Coefficient", "Global Moran's I"), horizontal_spacing=0.12)
    fig_ts.add_trace(go.Scatter(x=ts_df['year'], y=ts_df['gini'], mode='lines+markers',
        line=dict(color='#4a6fa5', width=2.5), marker=dict(size=10)), row=1, col=1)
    fig_ts.add_trace(go.Scatter(x=ts_df['year'], y=ts_df['morans_i'], mode='lines+markers',
        line=dict(color='#c23a3a', width=2.5), marker=dict(size=10)), row=1, col=2)
    fig_ts.update_layout(**base_layout(), showlegend=False)
    fig_ts.update_xaxes(tickmode='array', tickvals=ts_df['year'].tolist(), title_text='January of Each Year')
    fig_ts.update_yaxes(title_text='Gini Coefficient', col=1)
    fig_ts.update_yaxes(title_text="Moran's I", col=2)
    save_html(fig_ts, 'ext2_timeseries_gini_moran.html')

    fig_share = go.Figure(go.Scatter(x=ts_df['year'], y=ts_df['uber_share'],
        mode='lines+markers', line=dict(color='#3d4f5f', width=2.5), marker=dict(size=10)))
    fig_share.update_layout(**base_layout(),
        xaxis=styled_axis(title_text='January of Each Year', tickmode='array', tickvals=ts_df['year'].tolist()),
        yaxis=styled_axis(title_text='Uber Market Share (%)', range=[0, 100]), showlegend=False)
    save_html(fig_share, 'ext2_uber_share_timeseries.html')
else:
    print("  Only 2 data points. Skipping time series plots.")


# %% [markdown]
# ---
# ## 25. Summary

# %%
print("\n" + "=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)
print(f"  Market share: {100 * uber_n_2018 / total_2018:.1f}% -> {100 * uber_n_2025 / total_2025:.1f}%")
print(f"  Gini: {gini_18:.3f} [{ci_2018[0]:.3f}, {ci_2018[1]:.3f}] -> {gini_25:.3f} [{ci_2025[0]:.3f}, {ci_2025[1]:.3f}]")
print(f"  Moran's I: {mi_2018:.4f} -> {mi_2025:.4f}")
print(f"  Avg cluster shift: {avg_shift:.2f} km")
print(f"  Intra-cluster trips: {intra_2018_pct:.1f}% -> {intra_2025_pct:.1f}%")
print(f"  Zone localization: {mean_local_2018:.1f}% -> {mean_local_2025:.1f}%")
print(f"  Aggregate JSD: {agg_jsd:.6f}")
print(f"  KS mismatch: D={ks_stat:.4f}, p={ks_p:.2e}")
print(f"  Levene: W={lev_stat:.4f}, p={lev_p:.2e}")
print("=" * 70)

all_outputs = [
    'cluster_map_2018.html', 'cluster_map_2025.html',
    'fix_cluster_shifts.html', 'pickup_density_change.html',
    'fix_borough_pct.html', 'fix_hourly_patterns.html',
    'fix_cluster_heatmap_changes.html', 'fix_hourly_composition.html',
    'lorenz_curve.html', 'lisa_map_2018.html', 'lisa_map_2025.html',
    'ext1_market_share_breakdown.html', 'ext1_market_share_stacked.html',
    'ext3_od_sankey_2018.html', 'ext3_od_sankey_2025.html',
    'fix_od_change.html', 'ext3_dropoff_density_change.html',
    'fix_trip_distance.html', 'fix_distance_by_cluster.html',
    'fix_jsd_by_cluster.html', 'fix_localization_pct.html',
    'ext6_localization_change_map.html',
    'fix_demand_profiles_2018.html', 'fix_demand_profiles_2025.html',
    'fix_demand_map_2018.html', 'fix_demand_map_2025.html',
    'ext8_graduated_scorecard.html', 'ext9_bootstrap_gini.html',
    'ext10_mismatch_ratios.html', 'ext10_mismatch_change_map.html',
]
print(f"\nOutputs ({len(all_outputs)} files):")
for f in all_outputs:
    status = "OK" if os.path.exists(OUTPUT_DIR + f) else "MISSING"
    print(f"  [{status}] {f}")

Configuration:
  Sample size: 20,000,000 trips per year
  Clusters: 6
  Output: /Users/leoss/Desktop/Portfolio/Website-/projects/uber/outputs/
Loaded 263 zone centroids
Loaded 263 taxi zone geometries
Spatial weights: KNN k=6, 263 geometries
K-nearest zones: K=5, 260 zones

[Loading 2018] fhv_tripdata_2018-01.parquet
  Total rows: 19,808,094
  Uber trips: 4,502,999 (22.7%)
  Final sample: 4,519,809 trips
    Manhattan: Gramercy:  1,883,665 ( 41.7%)
    Queens: Jamaica:    365,597 (  8.1%)
    Bronx: East Tremont:    477,684 ( 10.6%)
    Brooklyn: Borough Park:    388,956 (  8.6%)
    Manhattan: Yorkville East:    776,981 ( 17.2%)
    Brooklyn: Ocean Hill:    626,926 ( 13.9%)

[Loading 2025] fhvhv_tripdata_2025-01.parquet
  Total rows: 20,405,666
  Uber trips: 15,356,455 (75.3%)


/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_56224/2623011230.py:495: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<IntegerArray>
[211, 181,  76,  76,  89, 235, 242, 226, 260, 265,
 ...
 243, 132,  79, 113, 144, 148, 242, 235, 235, 133]
Length: 15401331, dtype: Int64' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.



  Final sample: 15,447,196 trips
    Brooklyn: Prospect-Lefferts Gardens:  3,282,505 ( 21.2%)
    Manhattan: Murray Hill:  5,992,333 ( 38.8%)
    Queens: Elmhurst:  1,857,722 ( 12.0%)
    Queens: Baisley Park:  1,358,465 (  8.8%)
    Bronx: Claremont/Bathgate:  2,714,803 ( 17.6%)
    Staten Island: Westerleigh:    241,368 (  1.6%)

Computing trip distances...
  2018: 4,364,044 trips with distances
  2025: 14,804,246 trips with distances
Cluster matches (2018 -> 2025):
  Manhattan: Gramercy -> Manhattan: Murray Hill (1.22 km)
  Queens: Jamaica -> Queens: Baisley Park (2.43 km)
  Bronx: East Tremont -> Bronx: Claremont/Bathgate (0.72 km)
  Brooklyn: Borough Park -> Staten Island: Westerleigh (11.62 km)
  Manhattan: Yorkville East -> Queens: Elmhurst (6.07 km)
  Brooklyn: Ocean Hill -> Brooklyn: Prospect-Lefferts Gardens (3.03 km)

Gini: 0.5109 (2018) -> 0.4401 (2025)
Bootstrapping Gini CIs...
  2018: 0.5109 [0.4754, 0.5414]
  2025: 0.4401 [0.4016, 0.4731]
Moran's I: 0.5409 (p=0.0010) -> 

In [8]:
# ── FIX: Market Share Stacked (legend overlap) ───────────────────────────
# Paste into a cell after the main notebook has run

fig_ms_stack = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Absolute Trip Counts', 'Market Share (%)'),
    horizontal_spacing=0.15,
)

for op in ['Uber', 'Lyft', 'Other FHV']:
    subset = ms_data[ms_data['Operator'] == op]
    fig_ms_stack.add_trace(go.Bar(
        x=subset['Year'], y=subset['Trips'], name=op,
        marker_color=OPERATOR_COLORS[op], legendgroup=op, showlegend=True,
        hovertemplate=f'<b>{op}</b><br>Trips: %{{y:,}}<extra></extra>',
    ), row=1, col=1)
    fig_ms_stack.add_trace(go.Bar(
        x=subset['Year'], y=subset['Share'], name=op,
        marker_color=OPERATOR_COLORS[op], legendgroup=op, showlegend=False,
        hovertemplate=f'<b>{op}</b><br>Share: %{{y:.1f}}%<extra></extra>',
    ), row=1, col=2)

fig_ms_stack.update_layout(
    **base_layout(),
    barmode='stack',
    legend=dict(
        orientation='h',
        yanchor='bottom', y=1.08,
        xanchor='center', x=0.5,
        font=dict(size=STYLE['legend_size'], family=STYLE['font_family']),
    ),
)
fig_ms_stack.update_yaxes(title_text='Total Trips', col=1)
fig_ms_stack.update_yaxes(title_text='Share (%)', col=2)
save_html(fig_ms_stack, 'ext1_market_share_stacked.html')

  Saved: ext1_market_share_stacked.html (standalone)


In [9]:
# ── FIX: Centroid Shifts (show cluster zones, colored lines, clean legend) ─
# Paste into a cell after the main notebook has run

fig_shift = go.Figure()
add_bg_layer(fig_shift)

# Layer 1: 2018 cluster zones as semi-transparent colored fills
zone_ids_set_2018 = set(zone_clusters_2018['zone_id'].values)
fgeo_2018 = filter_geojson(zone_ids_set_2018)

for i in range(N_CLUSTERS):
    subset = zone_clusters_2018[zone_clusters_2018['cluster'] == i]
    if len(subset) == 0:
        continue
    subset_ids = set(subset['zone_id'].values)
    sgeo = {
        'type': 'FeatureCollection',
        'features': [f for f in fgeo_2018['features']
                     if f['properties']['LocationID'] in subset_ids]
    }
    fig_shift.add_trace(go.Choroplethmap(
        geojson=sgeo,
        locations=subset['zone_id'].values,
        featureidkey='properties.LocationID',
        z=[1] * len(subset),
        colorscale=[[0, STYLE['cluster_colors'][i]], [1, STYLE['cluster_colors'][i]]],
        marker_opacity=0.25,
        marker_line_width=0.3,
        marker_line_color='rgba(255,255,255,0.5)',
        showscale=False, hoverinfo='skip', showlegend=False,
    ))

# Layer 2: shift lines (colored by cluster, not black)
for sa in shift_data:
    lat1, lon1 = centers_2018[sa['idx_18']]
    lat2, lon2 = centers_2025[sa['idx_25']]
    color = STYLE['cluster_colors'][sa['idx_18'] % len(STYLE['cluster_colors'])]
    fig_shift.add_trace(go.Scattermap(
        lat=[lat1, lat2], lon=[lon1, lon2],
        mode='lines', line=dict(width=3, color=color),
        showlegend=False,
        hovertemplate=(
            f"<b>{short_labels_2018[sa['idx_18']]}</b><br>"
            f"Shift: {sa['dist_km']:.2f} km<extra></extra>"
        ),
    ))

# ── FIX: Centroid Shifts (highlight cluster acquisitions) ─────────────────
# Paste into a cell after the main notebook has run
#
# Logic:
#   - Every zone colored by its 2025 cluster assignment
#   - Zones that STAYED in the same matched cluster: faint (opacity 0.15)
#   - Zones that SWITCHED cluster between years: strong (opacity 0.65)
#   - Shift lines + centroids overlaid on top

print("Centroid shifts (acquisition view)...")

# Build zone-level comparison: did each zone change cluster?
zc18 = zone_clusters_2018[['zone_id', 'cluster']].rename(columns={'cluster': 'c18'})
zc25 = zone_clusters_2025[['zone_id', 'cluster']].rename(columns={'cluster': 'c25'})
zone_comp = zc18.merge(zc25, on='zone_id', how='outer')

# Map 2018 cluster to its expected 2025 match
zone_comp['c18_matched'] = zone_comp['c18'].map(cluster_matches)
# Zone switched if its actual 2025 cluster differs from the matched expectation
zone_comp['switched'] = zone_comp['c25'] != zone_comp['c18_matched']
# Zones only in one year: treat as acquired
zone_comp['switched'] = zone_comp['switched'].fillna(True)

# Merge 2025 cluster info for hover
zone_comp = zone_comp.merge(
    zone_clusters_2025[['zone_id', 'cluster_name', 'zone_name', 'borough']],
    on='zone_id', how='left'
)

fig_shift = go.Figure()
add_bg_layer(fig_shift)

# Layer 1: zones colored by 2025 cluster, opacity by switched/stayed
all_zone_ids_with_data = set(zone_comp.dropna(subset=['c25'])['zone_id'].values)
fgeo_all = filter_geojson(all_zone_ids_with_data)

for i25 in range(N_CLUSTERS):
    # Find the 2018 index this 2025 cluster maps back to (for consistent color)
    matched_18 = [k for k, v in cluster_matches.items() if v == i25]
    color_idx = matched_18[0] if matched_18 else i25
    color = STYLE['cluster_colors'][color_idx]

    for is_switched, opacity in [(False, 0.15), (True, 0.65)]:
        subset = zone_comp[
            (zone_comp['c25'] == i25) & (zone_comp['switched'] == is_switched)
        ]
        if len(subset) == 0:
            continue

        subset_ids = set(subset['zone_id'].values)
        sgeo = {
            'type': 'FeatureCollection',
            'features': [f for f in fgeo_all['features']
                         if f['properties']['LocationID'] in subset_ids]
        }

        if is_switched:
            fig_shift.add_trace(go.Choroplethmap(
                geojson=sgeo,
                locations=subset['zone_id'].values,
                featureidkey='properties.LocationID',
                z=[1] * len(subset),
                colorscale=[[0, color], [1, color]],
                marker_opacity=opacity,
                marker_line_width=0.4,
                marker_line_color='rgba(255,255,255,0.6)',
                showscale=False, showlegend=False,
                customdata=np.column_stack([
                    subset['zone_name'].fillna('Unknown').values,
                    subset['borough'].fillna('Unknown').values,
                ]),
                hovertemplate=(
                    '<b>%{customdata[0]}</b> (%{customdata[1]})<br>'
                    f'Acquired by: {short_labels_2018.get(color_idx, "")}'
                    '<extra></extra>'
                ),
            ))
        else:
            fig_shift.add_trace(go.Choroplethmap(
                geojson=sgeo,
                locations=subset['zone_id'].values,
                featureidkey='properties.LocationID',
                z=[1] * len(subset),
                colorscale=[[0, color], [1, color]],
                marker_opacity=opacity,
                marker_line_width=0.2,
                marker_line_color='rgba(200,200,200,0.3)',
                showscale=False, showlegend=False, hoverinfo='skip',
            ))

# Layer 2: shift lines (colored by cluster)
for sa in shift_data:
    lat1, lon1 = centers_2018[sa['idx_18']]
    lat2, lon2 = centers_2025[sa['idx_25']]
    color = STYLE['cluster_colors'][sa['idx_18'] % len(STYLE['cluster_colors'])]
    fig_shift.add_trace(go.Scattermap(
        lat=[lat1, lat2], lon=[lon1, lon2],
        mode='lines', line=dict(width=3.5, color=color),
        showlegend=False,
        hovertemplate=(
            f"<b>{short_labels_2018[sa['idx_18']]}</b><br>"
            f"Shift: {sa['dist_km']:.2f} km<extra></extra>"
        ),
    ))

# Layer 3: 2018 centroids (circles)
for i in range(N_CLUSTERS):
    fig_shift.add_trace(go.Scattermap(
        lat=[centers_2018[i, 0]], lon=[centers_2018[i, 1]],
        mode='markers',
        marker=dict(size=12, color=STYLE['cluster_colors'][i],
                    opacity=0.95, symbol='circle'),
        name=f'{short_labels_2018[i]} (2018)',
        legendgroup=short_labels_2018[i],
        hovertemplate=f'<b>2018: {short_labels_2018[i]}</b><extra></extra>',
    ))

# Layer 4: 2025 centroids (squares)
for i in range(N_CLUSTERS):
    matched_18 = [k for k, v in cluster_matches.items() if v == i]
    color_idx = matched_18[0] if matched_18 else i
    label_18 = short_labels_2018[color_idx]
    fig_shift.add_trace(go.Scattermap(
        lat=[centers_2025[i, 0]], lon=[centers_2025[i, 1]],
        mode='markers',
        marker=dict(size=12, color=STYLE['cluster_colors'][color_idx],
                    opacity=0.95, symbol='square'),
        name=f'{label_18} (2025)',
        legendgroup=label_18,
        hovertemplate=f'<b>2025: {short_labels_2025[i]}</b><extra></extra>',
    ))

# Annotation: how many zones switched
n_switched = int(zone_comp['switched'].sum())
n_total = len(zone_comp.dropna(subset=['c25']))
pct_switched = 100 * n_switched / n_total

fig_shift.update_layout(
    **base_layout(height=700, width=1100, margin=STYLE['margin_map']),
    map_style=STYLE['map_style'],
    map_zoom=9.8,
    map_center=dict(lat=40.72, lon=-73.94),
    legend=dict(
        orientation='v',
        yanchor='top', y=0.98,
        xanchor='left', x=0.01,
        bgcolor='rgba(255,255,255,0.92)',
        bordercolor='#dde1e7', borderwidth=1,
        font=dict(size=10, family=STYLE['font_family']),
        itemsizing='constant',
        tracegroupgap=2,
    ),
    annotations=[dict(
        x=0.99, y=0.02, xref='paper', yref='paper',
        xanchor='right', yanchor='bottom',
        text=(f"<b>{n_switched}</b> of {n_total} zones switched cluster "
              f"({pct_switched:.0f}%)<br>"
              f"Strong fill = acquired | Faint = retained"),
        showarrow=False,
        font=dict(size=10, family=STYLE['font_family'], color='#555'),
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#ddd', borderwidth=1,
    )],
)
save_html(fig_shift, 'fix_cluster_shifts.html')
print(f"  {n_switched} of {n_total} zones switched cluster ({pct_switched:.0f}%)")

Centroid shifts (acquisition view)...
  Saved: fix_cluster_shifts.html (standalone)
  78 of 259 zones switched cluster (30%)
